In [1]:
import torch
import math
import numpy as np
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [2]:
# using pandas reading csv data and lablelling it

df = pd.read_csv('iris.csv')
df["species"] = df["species"].map({
    "setosa":0,
    "versicolor":1,
    "virginica":2
})


In [37]:
# assigning to x, y
x = df.drop("species", axis = 1)
y = df["species"]
x,y

(     sepal_length  sepal_width  petal_length  petal_width
 0             5.1          3.5           1.4          0.2
 1             4.9          3.0           1.4          0.2
 2             4.7          3.2           1.3          0.2
 3             4.6          3.1           1.5          0.2
 4             5.0          3.6           1.4          0.2
 ..            ...          ...           ...          ...
 145           6.7          3.0           5.2          2.3
 146           6.3          2.5           5.0          1.9
 147           6.5          3.0           5.2          2.0
 148           6.2          3.4           5.4          2.3
 149           5.9          3.0           5.1          1.8
 
 [150 rows x 4 columns],
 0      0
 1      0
 2      0
 3      0
 4      0
       ..
 145    2
 146    2
 147    2
 148    2
 149    2
 Name: species, Length: 150, dtype: int64)

In [38]:
# dividing into train, test data
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(df.dtypes)
print(df.head())

sepal_length    float64
sepal_width     float64
petal_length    float64
petal_width     float64
species           int64
dtype: object
   sepal_length  sepal_width  petal_length  petal_width  species
0           5.1          3.5           1.4          0.2        0
1           4.9          3.0           1.4          0.2        0
2           4.7          3.2           1.3          0.2        0
3           4.6          3.1           1.5          0.2        0
4           5.0          3.6           1.4          0.2        0


In [39]:
# standardization

x_train = (x_train - x_train.mean(axis=0)) / x_train.std(axis=0)
x_train
x_test = (x_test - x_test.mean(axis=0)) / x_train.std(axis=0)
x_test

,sepal_length,sepal_width,petal_length,petal_width
38,-1.45,-0.093333,-2.41,-0.976667
127,0.25,-0.093333,1.19,0.623333
57,-0.95,-0.693333,-0.41,-0.176667
93,-0.85,-0.793333,-0.41,-0.176667
42,-1.45,0.106667,-2.41,-0.976667
56,0.45,0.206667,0.99,0.423333
22,-1.25,0.506667,-2.71,-0.976667
20,-0.45,0.306667,-2.01,-0.976667
147,0.65,-0.093333,1.49,0.823333
84,-0.45,-0.093333,0.79,0.323333


In [40]:
# making as tensors

x_train_torch = torch.tensor(x_train.values, dtype=torch.float32)
x_test_torch = torch.tensor(x_test.values, dtype=torch.float32)

y_train_torch = torch.tensor(y_train.values, dtype=torch.long)

y_test_torch = torch.tensor(y_test.values, dtype=torch.long)

In [41]:
x_train_torch,x_test_torch, y_train_torch, y_test_torch

(tensor([[-1.7144, -0.3235, -1.3414, -1.3147],
         [-1.1198, -1.2210,  0.4126,  0.6491],
         [ 1.1396, -0.5479,  0.5823,  0.2564],
         [-1.1198,  0.1253, -1.2848, -1.4456],
         [-0.4063, -1.2210,  0.1297,  0.1255],
         [ 0.5450, -1.2210,  0.6955,  0.9110],
         [-0.2874, -0.7722,  0.2428,  0.1255],
         [ 0.5450, -0.5479,  0.7520,  0.3873],
         [ 2.2099, -0.0991,  1.3178,  1.4347],
         [ 2.2099,  1.6960,  1.6573,  1.3037],
         [ 2.0909, -0.0991,  1.6007,  1.1728],
         [ 0.1883, -0.3235,  0.4126,  0.3873],
         [-1.0009, -2.3429, -0.1532, -0.2673],
         [-0.0495, -0.7722,  0.1862, -0.2673],
         [-0.0495, -0.9966,  0.1297, -0.0055],
         [-1.3576,  0.3497, -1.2283, -1.3147],
         [-0.8820,  1.6960, -1.2848, -1.1837],
         [ 1.0207, -1.2210,  1.1481,  0.7801],
         [ 0.6640,  0.1253,  0.9784,  0.7801],
         [-0.5252,  0.7984, -1.2848, -1.0528],
         [ 0.5450, -1.2210,  0.6389,  0.3873],
         [-0.

In [42]:
# creating a neural network that initilizes weights automatically
# here 2 layers
# 1st layer -> relu -> layer2


class SimpleMLP(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(4, 8)
        self.layer2 = nn.Linear(8, 3)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        return self.layer2(x)

In [33]:
# creating a model
model = SimpleMLP()

In [34]:
# using crossentropy

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

In [44]:
train_dataset = TensorDataset(x_train_torch, y_train_torch)
test_dataset = TensorDataset(x_test_torch, y_test_torch)

train_loader = DataLoader(train_dataset, batch_size = 1, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False)

In [45]:
num_epochs = 2

for epoch in range(num_epochs):
    for i, (i,o) in enumerate(train_loader):
        y_hat = model(i)
        loss = criterion(y_hat, o)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

print('training done')
        

training done


In [46]:
model.eval()

SimpleMLP(
  (layer1): Linear(in_features=4, out_features=8, bias=True)
  (layer2): Linear(in_features=8, out_features=3, bias=True)
)

In [47]:
# testing
correct, total = 0,0
with torch.no_grad():
    for i, (i,o) in enumerate(test_loader):
        outputs = model(i)
        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == o).sum().item()
        total += o.size(0)
print('correct ones', correct)
print('total', total)

correct ones 27
total 30


In [48]:
import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce GTX 1630
